# Imports

In [29]:
import os

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, spearmanr, pointbiserialr
from scipy.stats import f_oneway, kruskal

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.metrics import r2_score, roc_curve, auc

from autogluon.tabular import TabularDataset, TabularPredictor
#from autogluon.multimodal import MultiModalPredictor
import xgboost as xgb

plt.style.use('default')
sns.set_palette("husl")
SEED = 43
np.random.seed(SEED)

VAL_TILE_FRACTION = 0.4
TILE_SIZE_DEG = 0.01  # approx 1km


SCORING = {
    'r2': 'r2',
    'mae': 'neg_mean_absolute_error',
    'mse': 'neg_mean_squared_error'
}
GRID_SEARCH_KWARGS = dict(
    cv=10,
    scoring=SCORING,
    refit='r2',
    n_jobs=-1,
    return_train_score=True
)
LINEAR_PARAM_GRID = {
    'model__fit_intercept': [True, False],
    'model__positive': [False, True]
}
TREE_PARAM_GRID = {
    'model__max_depth': [None, 10, 20],
    'model__min_samples_split': [1, 100],
    'model__min_samples_leaf': [2, 100],
    'model__max_features': [None, 'sqrt', 'log2']
}
RF_PARAM_GRID = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [None, 10, 20],
    'model__min_samples_split': [1, 100],
    'model__min_samples_leaf': [2, 100],
    'model__max_features': [None, 'sqrt', 'log2']
}
MLP_PARAM_GRID = {
    'model__hidden_layer_sizes': [(100,), (100, 50), (50, 25)],
    'model__activation': ['relu', 'tanh'],
    'model__alpha': [0.0001, 0.001, 0.01],
    'model__learning_rate_init': [0.001, 0.01]
}
XGB_PARAM_GRID = {
    'model__learning_rate': [0.01, 0.1, 0.2],
    'model__n_estimators': [50, 100, 200]
}

def evaluate_regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    errors = y_true - y_pred

    bin_size = 0.1
    bins = np.arange(0.0, 1.0 + bin_size, bin_size)
    bin_ids = np.digitize(y_pred, bins, right=False)

    ece = 0.0
    N = len(y_pred)

    for i in range(1, len(bins)):
        mask = bin_ids == i
        if not np.any(mask):
            continue

        bin_center = (bins[i - 1] + bins[i]) / 2.0
        median_obs = np.median(y_true[mask])
        ece += (mask.sum() / N) * abs(median_obs - bin_center)
        
    return {
        'r2': r2_score(y_true, y_pred),
        'mae': float(np.mean(np.abs(errors))),
        'mse': float(np.mean(errors ** 2)),
        "p95": np.quantile(np.abs(errors), 0.95),
        'ece': float(ece),              # Expected calibration error
    }


def record_regression_results(results, target, model_name, y_true, y_pred, split='test'):
    metrics = evaluate_regression_metrics(y_true, y_pred)
    split_label = split.capitalize()
    print(f"{split_label} R2 score {metrics['r2']:.4f}")
    print(f"{split_label} MAE score {metrics['mae']:.4f}")
    print(f"{split_label} MSE score {metrics['mse']:.4f}")
    print(f"{split_label} P95 score {metrics['p95']:.4f}")
    print(f"{split_label} ECE score {metrics['ece']:.4f}")
    results.append({
        'target': target,
        'model': model_name,
        f'{split}_r2': metrics['r2'],
        f'{split}_mae': metrics['mae'],
        #f'{split}_mse': metrics['mse'],
        #f'{split}_p95': metrics['p95'],
        f'{split}_ece': metrics['ece'],
    })
    return metrics


# Load data

In [30]:
data_path_file = "faster-rcnn_metafeatures.csv"
data = pd.read_csv("../data/" + data_path_file, index_col=0)

print(data)

      country time_of_day        lat       long       road_type  \
1          PL         day  52.249511  21.043137            city   
6          PL         day  52.239332  21.030383            city   
39         PL         day  52.234241  21.003985            city   
49         PL         day  52.224666  21.071192            city   
52         PL    twilight  52.231355  21.054809            city   
...       ...         ...        ...        ...             ...   
99902      SE         day  63.178756  14.690088   smaller-rural   
99904      PL         day  54.353453  18.645450  arterial-urban   
99939      DE       night  50.116028   8.622851         highway   
99941      DE         day  53.057472   8.958626  arterial-urban   
99997      SE    twilight  59.103593  12.789612  arterial-rural   

      road_condition              weather  solar_angle_elevation  month  hour  \
1             normal    partly-cloudy-day              51.723833      4    10   
6             normal            c

In [31]:
data = data.dropna().reset_index(drop=True)
num_rows, num_cols = data.shape
print(f"After dropping missing values: {num_rows:,} rows and {num_cols} columns")

After dropping missing values: 9,521 rows and 38 columns


In [32]:
iou = data["iou"]
lrp = data["lrp"]
conf = data["conf"]
image_paths_dict = {int(img_pth.split("_")[0]):f"../data/zod_yolo/images/test/{img_pth}" for img_pth in os.listdir("../data/zod_yolo/images/test") if img_pth.endswith(".jpg")}
img_path = pd.DataFrame.from_dict(image_paths_dict, orient='index', columns=["Images"])

data = data.drop(columns=["iou", "lrp"])
data_indices = data.index.to_numpy()

In [33]:
all_columns = data.columns    
all_columns = all_columns.drop(["weather_code", "cloud_cover_low", "cloud_cover_mid"])

print(f"Number of rows: {data.shape[0]}")
print(f"Columns: {all_columns.sort_values()}")
print(f"Number of columns: {len(all_columns)}")

Number of rows: 9521
Columns: Index(['brightness', 'camera_distance_from_ground', 'camera_offset',
       'camera_pitch_angle', 'cloud_cover', 'complexity', 'conf', 'contrast',
       'country', 'distortion_magnitude', 'field_view_horizontal',
       'forward_acceleration', 'forward_velocity', 'hour', 'laplacian', 'lat',
       'lateral_acceleration', 'lateral_velocity', 'long', 'month',
       'noisiness', 'quality', 'rain', 'relative_humidity_2m',
       'road_condition', 'road_type', 'sharpness', 'snowfall',
       'solar_angle_elevation', 'temperature_2m', 'time_of_day', 'weather',
       'wind_speed_10m'],
      dtype='object')
Number of columns: 33


In [34]:
numeric_columns = ['conf', 'solar_angle_elevation', 'temperature_2m', 'relative_humidity_2m', 'rain', 'snowfall', 'cloud_cover', 'wind_speed_10m', 'forward_acceleration', 'lateral_acceleration', 'forward_velocity', 'lateral_velocity', 'field_view_horizontal', 'camera_distance_from_ground', 'camera_pitch_angle', 'distortion_magnitude', 'camera_offset', 'laplacian', 'brightness', 'contrast', 'sharpness', 'noisiness', 'quality', 'complexity']

categorical_columns = ["country", "time_of_day", "road_type", "road_condition", "weather"]
other = ["hour", "month", "lat", "long"]
print(sorted(numeric_columns + categorical_columns + other))
assert len(all_columns) == (len(categorical_columns) + len(numeric_columns) + len(other)), "Columns not match"

['brightness', 'camera_distance_from_ground', 'camera_offset', 'camera_pitch_angle', 'cloud_cover', 'complexity', 'conf', 'contrast', 'country', 'distortion_magnitude', 'field_view_horizontal', 'forward_acceleration', 'forward_velocity', 'hour', 'laplacian', 'lat', 'lateral_acceleration', 'lateral_velocity', 'long', 'month', 'noisiness', 'quality', 'rain', 'relative_humidity_2m', 'road_condition', 'road_type', 'sharpness', 'snowfall', 'solar_angle_elevation', 'temperature_2m', 'time_of_day', 'weather', 'wind_speed_10m']


# Assessor Tests

Split data into train 60% and validation 40%. To prevent leakage, the data is binned in buckets by lat/long in 1km. The split is done by buckets ensuring no leakage. 

In [35]:
#Val split
def tile_id(lat, lon, size_deg):
    lat_bin = np.floor(lat / size_deg) * size_deg
    lon_bin = np.floor(lon / size_deg) * size_deg
    return f'{lat_bin:.4f}_{lon_bin:.4f}'

In [36]:
## Make split of train into train and val by tiles
ids = data.index.tolist()
coords = {frame_id: {"lat": data.loc[frame_id, "lat"], "lon": data.loc[frame_id, "long"]} for frame_id in ids}
df_coords = pd.DataFrame.from_dict(coords, orient='index')
df_coords['tile_id'] = [tile_id(lat, lon, TILE_SIZE_DEG) for lat, lon in zip(df_coords['lat'], df_coords['lon'])]

unique_tiles = sorted(set(df_coords['tile_id'].dropna().unique()))
rng = np.random.default_rng(SEED)
val_tile_count = int(len(unique_tiles) * VAL_TILE_FRACTION)
val_tiles = set(rng.choice(unique_tiles, size=val_tile_count, replace=False))
df_coords['split'] = np.where(df_coords['tile_id'].isin(val_tiles), 'val', 'train')
test_idx = df_coords[df_coords['split'] == 'val'].index.tolist()
train_idx = df_coords[df_coords['split'] == 'train'].index.tolist()
print(f"Total train frames: {len(train_idx)}, val frames: {len(test_idx)}")

Total train frames: 5486, val frames: 4035


In [37]:
X_train = data.loc[train_idx]
y_train = iou.loc[train_idx]
y_train_lrp = lrp.loc[train_idx]
conf_train = conf.loc[train_idx]
conf_train = pd.DataFrame(conf_train)

X_test = data.loc[test_idx]
y_test = iou.loc[test_idx]
y_test_lrp = lrp.loc[test_idx]
conf_test = conf.loc[test_idx]
conf_test = pd.DataFrame(conf_test)

print("X:", len(X_train), len(X_test))
print("y:", len(y_train), len(y_test))
print("Conf:", len(conf_train), len(conf_test))
print(X_train.columns)

X: 5486 4035
y: 5486 4035
Conf: 5486 4035
Index(['country', 'time_of_day', 'lat', 'long', 'road_type', 'road_condition',
       'weather', 'solar_angle_elevation', 'month', 'hour',
       'forward_acceleration', 'lateral_acceleration', 'forward_velocity',
       'lateral_velocity', 'field_view_horizontal',
       'camera_distance_from_ground', 'camera_pitch_angle',
       'distortion_magnitude', 'camera_offset', 'laplacian', 'quality',
       'brightness', 'noisiness', 'sharpness', 'contrast', 'complexity',
       'temperature_2m', 'relative_humidity_2m', 'rain', 'snowfall',
       'cloud_cover', 'cloud_cover_low', 'cloud_cover_mid', 'wind_speed_10m',
       'weather_code', 'conf'],
      dtype='object')


In [38]:
model_results = []

### IoU

#### Baseline

Random Prediction. Mean of metric over training set.

In [39]:
mean_iou_train = np.mean(y_train)
iou_pred_test = np.full_like(y_test, mean_iou_train)
record_regression_results(model_results, 'IoU', 'Constant Mean Predictor', y_test, iou_pred_test)


Test R2 score -0.0005
Test MAE score 0.1601
Test MSE score 0.0436
Test P95 score 0.4752
Test ECE score 0.0282


{'r2': -0.0005354906844718954,
 'mae': 0.16010748262681426,
 'mse': 0.04361430177319595,
 'p95': 0.47519785040343304,
 'ece': 0.028198873996734652}

#### Linear Reg on Conf

In [40]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['conf']),
    ],
    remainder='passthrough'
)
linear_reg_conf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])
linear_reg_conf_grid = GridSearchCV(
    linear_reg_conf_pipeline,
    param_grid=LINEAR_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
linear_reg_conf_grid.fit(conf_train, y_train)

best_idx = linear_reg_conf_grid.best_index_
mean_train_r2 = linear_reg_conf_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = linear_reg_conf_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -linear_reg_conf_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -linear_reg_conf_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -linear_reg_conf_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -linear_reg_conf_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {linear_reg_conf_grid.best_params_}")
best_linear_reg_conf = linear_reg_conf_grid.best_estimator_


Mean CV train r2 score 0.3058
Mean CV test r2 score 0.1715
Mean CV train MAE score 0.1334
Mean CV test MAE score 0.1350
Mean CV train MSE score 0.0299
Mean CV test MSE score 0.0306
Best params: {'model__fit_intercept': True, 'model__positive': False}


In [41]:
y_pred_iou_baseline = best_linear_reg_conf.predict(conf_test)
record_regression_results(model_results, 'IoU', 'Univariate Linear Regression (Confidence)', y_test, y_pred_iou_baseline)


Test R2 score 0.2856
Test MAE score 0.1373
Test MSE score 0.0311
Test P95 score 0.3779
Test ECE score 0.0449


{'r2': 0.2855569755414872,
 'mae': 0.137267090988751,
 'mse': 0.031143256744617535,
 'p95': 0.37785774098620784,
 'ece': 0.04489003513017869}

#### Linear Regression

Train models with cv and then test.

In [42]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
linear_reg_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])
linear_reg_grid = GridSearchCV(
    linear_reg_pipeline,
    param_grid=LINEAR_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
linear_reg_grid.fit(X_train, y_train)

best_idx = linear_reg_grid.best_index_
mean_train_r2 = linear_reg_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = linear_reg_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -linear_reg_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -linear_reg_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -linear_reg_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -linear_reg_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {linear_reg_grid.best_params_}")
best_linear_reg = linear_reg_grid.best_estimator_

Mean CV train r2 score 0.4482
Mean CV test r2 score 0.3162
Mean CV train MAE score 0.1189
Mean CV test MAE score 0.1211
Mean CV train MSE score 0.0237
Mean CV test MSE score 0.0247
Best params: {'model__fit_intercept': True, 'model__positive': False}


In [43]:
y_pred_iou_lr = best_linear_reg.predict(X_test)
record_regression_results(model_results, 'IoU', 'Linear Regression', y_test, y_pred_iou_lr)


Test R2 score 0.4041
Test MAE score 0.1238
Test MSE score 0.0260
Test P95 score 0.3274
Test ECE score 0.0291


{'r2': 0.4040902188054042,
 'mae': 0.12379116075164405,
 'mse': 0.025976278971213945,
 'p95': 0.32744259211485843,
 'ece': 0.02912104610733577}

#### Decision Trees

In [44]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
decision_tree_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', DecisionTreeRegressor(random_state=SEED))
])
decision_tree_grid = GridSearchCV(
    decision_tree_pipeline,
    param_grid=TREE_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
decision_tree_grid.fit(X_train, y_train)

best_idx = decision_tree_grid.best_index_
mean_train_r2 = decision_tree_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = decision_tree_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -decision_tree_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -decision_tree_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -decision_tree_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -decision_tree_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {decision_tree_grid.best_params_}")
best_decision_tree = decision_tree_grid.best_estimator_


Mean CV train r2 score 0.4807
Mean CV test r2 score 0.3176
Mean CV train MAE score 0.1122
Mean CV test MAE score 0.1185
Mean CV train MSE score 0.0223
Mean CV test MSE score 0.0249
Best params: {'model__max_depth': None, 'model__max_features': None, 'model__min_samples_leaf': 100, 'model__min_samples_split': 100}


/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/model_selection/_validation.py:516: FitFailedWarning: 
180 fits failed out of a total of 360.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
180 fits failed with the following error:
Traceback (most recent call last):
  File "/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/model_selection/_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/base.py", line 1365, in wrapper
    return fit_method(estimator, *ar

In [45]:
y_pred_iou_dt = best_decision_tree.predict(X_test)
record_regression_results(model_results, 'IoU', 'Decision Tree', y_test, y_pred_iou_dt)


Test R2 score 0.4124
Test MAE score 0.1194
Test MSE score 0.0256
Test P95 score 0.3313
Test ECE score 0.0261


{'r2': 0.4123838056357124,
 'mae': 0.11939577932779473,
 'mse': 0.02561475356590142,
 'p95': 0.33127305033663373,
 'ece': 0.026070071145757635}

#### Random Forest

In [46]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
rand_forest_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=SEED, n_jobs=-1))
])
rand_forest_grid = GridSearchCV(
    rand_forest_pipeline,
    param_grid=RF_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
rand_forest_grid.fit(X_train, y_train)

best_idx = rand_forest_grid.best_index_
mean_train_r2 = rand_forest_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = rand_forest_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -rand_forest_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -rand_forest_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -rand_forest_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -rand_forest_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {rand_forest_grid.best_params_}")
best_rand_forest = rand_forest_grid.best_estimator_


/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/model_selection/_validation.py:516: FitFailedWarning: 
360 fits failed out of a total of 720.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
360 fits failed with the following error:
Traceback (most recent call last):
  File "/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/model_selection/_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/base.py", line 1365, in wrapper
    return fit_method(estimator, *ar

Mean CV train r2 score 0.5662
Mean CV test r2 score 0.3536
Mean CV train MAE score 0.1037
Mean CV test MAE score 0.1151
Mean CV train MSE score 0.0187
Mean CV test MSE score 0.0235
Best params: {'model__max_depth': 10, 'model__max_features': None, 'model__min_samples_leaf': 2, 'model__min_samples_split': 100, 'model__n_estimators': 100}


In [47]:
y_pred_iou_rf = best_rand_forest.predict(X_test)
record_regression_results(model_results, 'IoU', 'Random Forest', y_test, y_pred_iou_rf)


Test R2 score 0.4335
Test MAE score 0.1173
Test MSE score 0.0247
Test P95 score 0.3218
Test ECE score 0.0235


{'r2': 0.4334603230345887,
 'mae': 0.11731051863853675,
 'mse': 0.02469600795545461,
 'p95': 0.32177673250316874,
 'ece': 0.023472318125312506}

#### MLP

In [48]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
mlp_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', MLPRegressor(max_iter=500, random_state=SEED))
])
mlp_grid = GridSearchCV(
    mlp_pipeline,
    param_grid=MLP_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
mlp_grid.fit(X_train, y_train)

best_idx = mlp_grid.best_index_
mean_train_r2 = mlp_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = mlp_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -mlp_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -mlp_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -mlp_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -mlp_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {mlp_grid.best_params_}")
best_mlp = mlp_grid.best_estimator_


Mean CV train r2 score 0.5304
Mean CV test r2 score 0.3023
Mean CV train MAE score 0.1084
Mean CV test MAE score 0.1204
Mean CV train MSE score 0.0202
Mean CV test MSE score 0.0255
Best params: {'model__activation': 'tanh', 'model__alpha': 0.01, 'model__hidden_layer_sizes': (50, 25), 'model__learning_rate_init': 0.001}


In [49]:
y_pred_iou_mlp = best_mlp.predict(X_test)
record_regression_results(model_results, 'IoU', 'MLP', y_test, y_pred_iou_mlp)


Test R2 score 0.3960
Test MAE score 0.1219
Test MSE score 0.0263
Test P95 score 0.3308
Test ECE score 0.0271


{'r2': 0.3959527844268248,
 'mae': 0.1218919234311263,
 'mse': 0.026330997541370957,
 'p95': 0.33075147382739783,
 'ece': 0.02711067911101003}

#### XGBoost

In [50]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
xgb_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', xgb.XGBRegressor())
])
xgb_grid = GridSearchCV(
    xgb_pipeline,
    param_grid=XGB_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
xgb_grid.fit(X_train, y_train)


best_idx = xgb_grid.best_index_
mean_train_r2 = xgb_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = xgb_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -xgb_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -xgb_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -xgb_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -xgb_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {xgb_grid.best_params_}")
best_xgb = xgb_grid.best_estimator_

Mean CV train r2 score 0.7220
Mean CV test r2 score 0.3363
Mean CV train MAE score 0.0853
Mean CV test MAE score 0.1163
Mean CV train MSE score 0.0120
Mean CV test MSE score 0.0242
Best params: {'model__learning_rate': 0.1, 'model__n_estimators': 50}


In [51]:
y_pred_iou_xgb = best_xgb.predict(X_test)
record_regression_results(model_results, 'IoU', 'XGBoost', y_test, y_pred_iou_xgb)


Test R2 score 0.4168
Test MAE score 0.1194
Test MSE score 0.0254
Test P95 score 0.3337
Test ECE score 0.0223


{'r2': 0.41681923147234246,
 'mae': 0.11938096805279429,
 'mse': 0.02542140909913086,
 'p95': 0.33371342420578004,
 'ece': 0.02230624515854693}

#### Autogluon

##### Tabluar

In [52]:
train = pd.merge(X_train, y_train, left_index=True, right_index=True)

In [53]:
autoglue_model = TabularPredictor(label="iou", problem_type="regression", eval_metric="r2", path="../models/assessors/autoglue_tab/iou/")
autoglue_predictor = autoglue_model.fit(train)


Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.11.14
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP PREEMPT_DYNAMIC Debian 6.12.48-1 (2025-09-20)
CPU Count:          32
Pytorch Version:    2.9.1+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 23.56/23.56 GB
Total GPU Memory:   Free: 23.56 GB, Allocated: 0.00 GB, Total: 23.56 GB
GPU Count:          1
Memory Avail:       5.21 GB / 30.47 GB (17.1%)
Disk Space Avail:   1566.23 GB / 3542.28 GB (44.2%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='extreme'  : New in v1.5: The state-of-the-art for tabular data. Massively better than 'best' on datasets <100000 samples by using new Tabular 

In [54]:
autoglue_predictor.fit_summary(verbosity=2)

*** Summary of fit() ***
Estimated performance of each model:
                 model  score_val eval_metric  pred_time_val  fit_time  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
0  WeightedEnsemble_L2   0.481750          r2       0.014279  7.076225                0.000301           0.020717            2       True          9
1             CatBoost   0.475009          r2       0.003392  0.930872                0.003392           0.930872            1       True          4
2           LightGBMXT   0.467929          r2       0.001523  0.650235                0.001523           0.650235            1       True          1
3             LightGBM   0.464405          r2       0.001720  0.538851                0.001720           0.538851            1       True          2
4       NeuralNetTorch   0.461043          r2       0.009063  5.474400                0.009063           5.474400            1       True          7
5              XGBoost   0.439295          r

/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/autogluon/core/utils/plots.py:169: UserWarning: AutoGluon summary plots cannot be created because bokeh is not installed. To see plots, please do: "pip install bokeh==2.0.1"
  warnings.warn('AutoGluon summary plots cannot be created because bokeh is not installed. To see plots, please do: "pip install bokeh==2.0.1"')


{'model_types': {'LightGBMXT': 'LGBModel',
  'LightGBM': 'LGBModel',
  'RandomForestMSE': 'RFModel',
  'CatBoost': 'CatBoostModel',
  'ExtraTreesMSE': 'XTModel',
  'XGBoost': 'XGBoostModel',
  'NeuralNetTorch': 'TabularNeuralNetTorchModel',
  'LightGBMLarge': 'LGBModel',
  'WeightedEnsemble_L2': 'WeightedEnsembleModel'},
 'model_performance': {'LightGBMXT': 0.46792910601760396,
  'LightGBM': 0.46440485877704507,
  'RandomForestMSE': 0.42302976984247187,
  'CatBoost': 0.4750093726929012,
  'ExtraTreesMSE': 0.4279157779596088,
  'XGBoost': 0.43929538293561987,
  'NeuralNetTorch': 0.461042898751859,
  'LightGBMLarge': 0.4250663576038216,
  'WeightedEnsemble_L2': 0.48175043967310494},
 'model_best': 'WeightedEnsemble_L2',
 'model_paths': {'LightGBMXT': ['LightGBMXT'],
  'LightGBM': ['LightGBM'],
  'RandomForestMSE': ['RandomForestMSE'],
  'CatBoost': ['CatBoost'],
  'ExtraTreesMSE': ['ExtraTreesMSE'],
  'XGBoost': ['XGBoost'],
  'NeuralNetTorch': ['NeuralNetTorch'],
  'LightGBMLarge': ['Li

In [55]:
y_pred_iou_autg = autoglue_predictor.predict(X_test)
record_regression_results(model_results, 'IoU', 'Autogluon_Tab', y_test, y_pred_iou_autg)


Test R2 score 0.4466
Test MAE score 0.1167
Test MSE score 0.0241
Test P95 score 0.3192
Test ECE score 0.0204


{'r2': 0.44657479029994274,
 'mae': 0.1167180850489519,
 'mse': 0.02412433574769059,
 'p95': 0.31919693549474076,
 'ece': 0.020364911627706594}

##### Images

In [56]:
X_train_img = pd.merge(X_train, img_path, left_index=True, right_index=True)
X_test_img = pd.merge(X_test, img_path, how="left", right_index=True, left_index=True)
train_img = pd.merge(X_train_img, y_train, left_index=True, right_index=True)

In [57]:
#autoglue_model_img = MultiModalPredictor(label="iou", problem_type="regression", eval_metric="r2", path="../models/assessors/autoglue_tab/iou/img/")
hyperparams = {
    "env.strategy": "ddp_notebook",
    "env.num_gpus": 1,
    "env.num_workers": 0,
    #"optim.max_epochs": 15,
    #"optim.patience": 3,
    #"optim.learning_rate": 1e-4,
    #"optim.weight_decay": 1e-4,
    #"optim.warmup_steps": 500,
    #"env.per_gpu_batch_size": 8,
    #"env.batch_size": 128,
    #"model.timm_image.checkpoint_name": "convnext_base",
}
#autoglue_predictor_img = autoglue_model_img.fit(train_img, hyperparameters=hyperparams)

In [58]:
#autoglue_predictor_img.fit_summary(verbosity=2)

In [59]:
#y_pred = autoglue_predictor_img.predict(X_test_img)
#record_regression_results(model_results, 'IoU', 'Autogluon_Mult', y_test, y_pred)


### LRP


#### Baseline

Predict Only the Mean


In [60]:
mean_lrp_train = np.mean(y_train_lrp)
lrp_pred_test = np.full_like(y_test_lrp, mean_lrp_train)
record_regression_results(model_results, 'LRP', 'Constant Mean Predictor', y_test_lrp, lrp_pred_test)


Test R2 score -0.0000
Test MAE score 0.1391
Test MSE score 0.0345
Test P95 score 0.4016
Test ECE score 0.0107


{'r2': -5.200389865400723e-07,
 'mae': 0.13909003392535774,
 'mse': 0.034510716718885996,
 'p95': 0.40160728913619487,
 'ece': 0.010674297809600913}

#### Linear Reg on Conf

In [61]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['conf']),
    ],
    remainder='passthrough'
)
linear_reg_conf_lrp_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])
linear_reg_conf_lrp_grid = GridSearchCV(
    linear_reg_conf_lrp_pipeline,
    param_grid=LINEAR_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
linear_reg_conf_lrp_grid.fit(conf_train, y_train_lrp)

best_idx = linear_reg_conf_lrp_grid.best_index_
mean_train_r2 = linear_reg_conf_lrp_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = linear_reg_conf_lrp_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -linear_reg_conf_lrp_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -linear_reg_conf_lrp_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -linear_reg_conf_lrp_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -linear_reg_conf_lrp_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {linear_reg_conf_lrp_grid.best_params_}")
best_linear_reg_conf_lrp = linear_reg_conf_lrp_grid.best_estimator_


/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Mean CV train r2 score 0.3097
Mean CV test r2 score 0.1837
Mean CV train MAE score 0.1143
Mean CV test MAE score 0.1157
Mean CV train MSE score 0.0236
Mean CV test MSE score 0.0241
Best params: {'model__fit_intercept': True, 'model__positive': False}


In [62]:
y_pred_lrp_baseline = best_linear_reg_conf_lrp.predict(conf_test)
record_regression_results(model_results, 'LRP', 'Univariate Linear Regression (Confidence)', y_test_lrp, y_pred_lrp_baseline)


Test R2 score 0.2846
Test MAE score 0.1190
Test MSE score 0.0247
Test P95 score 0.3301
Test ECE score 0.0430


{'r2': 0.284604262708844,
 'mae': 0.11904875160718247,
 'mse': 0.02468880679241161,
 'p95': 0.3300741034655617,
 'ece': 0.04303243986588283}

#### Linear Regression


Train models with cv and then test.


In [63]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
linear_reg_lrp_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])
linear_reg_lrp_grid = GridSearchCV(
    linear_reg_lrp_pipeline,
    param_grid=LINEAR_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
linear_reg_lrp_grid.fit(X_train, y_train_lrp)

best_idx = linear_reg_lrp_grid.best_index_
mean_train_r2 = linear_reg_lrp_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = linear_reg_lrp_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -linear_reg_lrp_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -linear_reg_lrp_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -linear_reg_lrp_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -linear_reg_lrp_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {linear_reg_lrp_grid.best_params_}")
best_linear_reg_lrp = linear_reg_lrp_grid.best_estimator_


Mean CV train r2 score 0.4525
Mean CV test r2 score 0.3275
Mean CV train MAE score 0.1023
Mean CV test MAE score 0.1043
Mean CV train MSE score 0.0187
Mean CV test MSE score 0.0195
Best params: {'model__fit_intercept': True, 'model__positive': False}


In [64]:
y_pred_lrp_lr = best_linear_reg_lrp.predict(X_test)
record_regression_results(model_results, 'LRP', 'Linear Regression', y_test_lrp, y_pred_lrp_lr)


Test R2 score 0.4075
Test MAE score 0.1074
Test MSE score 0.0204
Test P95 score 0.3078
Test ECE score 0.0220


{'r2': 0.40752057056550095,
 'mae': 0.107352199556831,
 'mse': 0.02044687911780691,
 'p95': 0.30783888476035787,
 'ece': 0.022009551525116}

#### Decision Trees


In [65]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
decision_tree_lrp_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', DecisionTreeRegressor(random_state=SEED))
])
decision_tree_lrp_grid = GridSearchCV(
    decision_tree_lrp_pipeline,
    param_grid=TREE_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
decision_tree_lrp_grid.fit(X_train, y_train_lrp)

best_idx = decision_tree_lrp_grid.best_index_
mean_train_r2 = decision_tree_lrp_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = decision_tree_lrp_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -decision_tree_lrp_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -decision_tree_lrp_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -decision_tree_lrp_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -decision_tree_lrp_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {decision_tree_lrp_grid.best_params_}")
best_decision_tree_lrp = decision_tree_lrp_grid.best_estimator_


Mean CV train r2 score 0.5254
Mean CV test r2 score 0.3744
Mean CV train MAE score 0.0929
Mean CV test MAE score 0.0980
Mean CV train MSE score 0.0162
Mean CV test MSE score 0.0180
Best params: {'model__max_depth': None, 'model__max_features': None, 'model__min_samples_leaf': 100, 'model__min_samples_split': 100}


/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/model_selection/_validation.py:516: FitFailedWarning: 
180 fits failed out of a total of 360.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
180 fits failed with the following error:
Traceback (most recent call last):
  File "/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/model_selection/_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/base.py", line 1365, in wrapper
    return fit_method(estimator, *ar

In [66]:
y_pred_lrp_dt = best_decision_tree_lrp.predict(X_test)
record_regression_results(model_results, 'LRP', 'Decision Tree', y_test_lrp, y_pred_lrp_dt)


Test R2 score 0.4428
Test MAE score 0.1008
Test MSE score 0.0192
Test P95 score 0.2938
Test ECE score 0.0210


{'r2': 0.4428013196058578,
 'mae': 0.10079426946231274,
 'mse': 0.019229315815225433,
 'p95': 0.29380415074760297,
 'ece': 0.02097950223357852}

#### Random Forest


In [67]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
rand_forest_lrp_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=SEED, n_jobs=-1))
])
rand_forest_lrp_grid = GridSearchCV(
    rand_forest_lrp_pipeline,
    param_grid=RF_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
rand_forest_lrp_grid.fit(X_train, y_train_lrp)

best_idx = rand_forest_lrp_grid.best_index_
mean_train_r2 = rand_forest_lrp_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = rand_forest_lrp_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -rand_forest_lrp_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -rand_forest_lrp_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -rand_forest_lrp_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -rand_forest_lrp_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {rand_forest_lrp_grid.best_params_}")
best_rand_forest_lrp = rand_forest_lrp_grid.best_estimator_


/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/model_selection/_validation.py:516: FitFailedWarning: 
360 fits failed out of a total of 720.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
360 fits failed with the following error:
Traceback (most recent call last):
  File "/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/model_selection/_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/base.py", line 1365, in wrapper
    return fit_method(estimator, *ar

Mean CV train r2 score 0.6137
Mean CV test r2 score 0.4027
Mean CV train MAE score 0.0839
Mean CV test MAE score 0.0957
Mean CV train MSE score 0.0132
Mean CV test MSE score 0.0171
Best params: {'model__max_depth': 20, 'model__max_features': None, 'model__min_samples_leaf': 2, 'model__min_samples_split': 100, 'model__n_estimators': 200}


In [68]:
y_pred_lrp_rf = best_rand_forest_lrp.predict(X_test)
record_regression_results(model_results, 'LRP', 'Random Forest', y_test_lrp, y_pred_lrp_rf)


Test R2 score 0.4703
Test MAE score 0.0984
Test MSE score 0.0183
Test P95 score 0.2866
Test ECE score 0.0163


{'r2': 0.4702858779061935,
 'mae': 0.0983930236827102,
 'mse': 0.018280804502841703,
 'p95': 0.28659276383817384,
 'ece': 0.0163435724130203}

#### MLP


In [69]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
mlp_lrp_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', MLPRegressor(max_iter=500, random_state=SEED))
])
mlp_lrp_grid = GridSearchCV(
    mlp_lrp_pipeline,
    param_grid=MLP_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
mlp_lrp_grid.fit(X_train, y_train_lrp)

best_idx = mlp_lrp_grid.best_index_
mean_train_r2 = mlp_lrp_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = mlp_lrp_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -mlp_lrp_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -mlp_lrp_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -mlp_lrp_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -mlp_lrp_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {mlp_lrp_grid.best_params_}")
best_mlp_lrp = mlp_lrp_grid.best_estimator_


Mean CV train r2 score 0.5234
Mean CV test r2 score 0.3589
Mean CV train MAE score 0.0946
Mean CV test MAE score 0.0999
Mean CV train MSE score 0.0163
Mean CV test MSE score 0.0186
Best params: {'model__activation': 'relu', 'model__alpha': 0.001, 'model__hidden_layer_sizes': (50, 25), 'model__learning_rate_init': 0.01}


In [70]:
y_pred_lrp_mlp = best_mlp_lrp.predict(X_test)
record_regression_results(model_results, 'LRP', 'MLP', y_test_lrp, y_pred_lrp_mlp)


Test R2 score 0.4484
Test MAE score 0.1048
Test MSE score 0.0190
Test P95 score 0.2907
Test ECE score 0.0359


{'r2': 0.44842193210289705,
 'mae': 0.10476214746263222,
 'mse': 0.0190353445504261,
 'p95': 0.2907087077545132,
 'ece': 0.03591316278096026}

#### XGBoost

In [71]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
xgb_lrp_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', xgb.XGBRegressor())
])
xgb_lrp_grid = GridSearchCV(
    xgb_lrp_pipeline,
    param_grid=XGB_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
xgb_lrp_grid.fit(X_train, y_train_lrp)


best_idx = xgb_lrp_grid.best_index_
mean_train_r2 = xgb_lrp_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = xgb_lrp_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -xgb_lrp_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -xgb_lrp_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -xgb_lrp_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -xgb_lrp_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {xgb_lrp_grid.best_params_}")
best_xgb_lrp = xgb_lrp_grid.best_estimator_

Mean CV train r2 score 0.7547
Mean CV test r2 score 0.3936
Mean CV train MAE score 0.0700
Mean CV test MAE score 0.0961
Mean CV train MSE score 0.0084
Mean CV test MSE score 0.0175
Best params: {'model__learning_rate': 0.1, 'model__n_estimators': 50}


In [72]:
y_pred_lrp_xgb = best_xgb_lrp.predict(X_test)
record_regression_results(model_results, 'LRP', 'XGBoost', y_test_lrp, y_pred_lrp_xgb)


Test R2 score 0.4533
Test MAE score 0.0998
Test MSE score 0.0189
Test P95 score 0.2938
Test ECE score 0.0153


{'r2': 0.45326577205869645,
 'mae': 0.09981720063600533,
 'mse': 0.018868180248811837,
 'p95': 0.2937701463699339,
 'ece': 0.015281554404392262}

#### Autogluon

##### Tabluar

In [73]:
train_lrp = pd.merge(X_train, y_train_lrp, left_index=True, right_index=True)

In [74]:
autoglue_model_lrp = TabularPredictor(label="lrp", problem_type="regression", eval_metric="r2", path="../models/assessors/autoglue_tab/lrp/tab/")
autoglue_predictor_lrp = autoglue_model_lrp.fit(train_lrp)


Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.11.14
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP PREEMPT_DYNAMIC Debian 6.12.48-1 (2025-09-20)
CPU Count:          32
Pytorch Version:    2.9.1+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 23.56/23.56 GB
Total GPU Memory:   Free: 23.56 GB, Allocated: 0.00 GB, Total: 23.56 GB
GPU Count:          1
Memory Avail:       4.80 GB / 30.47 GB (15.8%)
Disk Space Avail:   1566.29 GB / 3542.28 GB (44.2%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='extreme'  : New in v1.5: The state-of-the-art for tabular data. Massively better than 'best' on datasets <100000 samples by using new Tabular 

In [75]:
autoglue_predictor_lrp.fit_summary(verbosity=2)

*** Summary of fit() ***
Estimated performance of each model:
                 model  score_val eval_metric  pred_time_val  fit_time  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
0  WeightedEnsemble_L2   0.522237          r2       0.016757  5.272604                0.000273           0.019239            2       True          9
1           LightGBMXT   0.513494          r2       0.001914  0.633114                0.001914           0.633114            1       True          1
2             CatBoost   0.511242          r2       0.003060  1.043845                0.003060           1.043845            1       True          4
3       NeuralNetTorch   0.507698          r2       0.011511  3.576405                0.011511           3.576405            1       True          7
4             LightGBM   0.501309          r2       0.001700  0.663152                0.001700           0.663152            1       True          2
5              XGBoost   0.480070          r

/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/autogluon/core/utils/plots.py:169: UserWarning: AutoGluon summary plots cannot be created because bokeh is not installed. To see plots, please do: "pip install bokeh==2.0.1"
  warnings.warn('AutoGluon summary plots cannot be created because bokeh is not installed. To see plots, please do: "pip install bokeh==2.0.1"')


{'model_types': {'LightGBMXT': 'LGBModel',
  'LightGBM': 'LGBModel',
  'RandomForestMSE': 'RFModel',
  'CatBoost': 'CatBoostModel',
  'ExtraTreesMSE': 'XTModel',
  'XGBoost': 'XGBoostModel',
  'NeuralNetTorch': 'TabularNeuralNetTorchModel',
  'LightGBMLarge': 'LGBModel',
  'WeightedEnsemble_L2': 'WeightedEnsembleModel'},
 'model_performance': {'LightGBMXT': 0.5134943622762544,
  'LightGBM': 0.5013086897929475,
  'RandomForestMSE': 0.4599587611291317,
  'CatBoost': 0.511241513550037,
  'ExtraTreesMSE': 0.4738103803738286,
  'XGBoost': 0.48006975722222056,
  'NeuralNetTorch': 0.5076980243271381,
  'LightGBMLarge': 0.4298410210234953,
  'WeightedEnsemble_L2': 0.5222366382850403},
 'model_best': 'WeightedEnsemble_L2',
 'model_paths': {'LightGBMXT': ['LightGBMXT'],
  'LightGBM': ['LightGBM'],
  'RandomForestMSE': ['RandomForestMSE'],
  'CatBoost': ['CatBoost'],
  'ExtraTreesMSE': ['ExtraTreesMSE'],
  'XGBoost': ['XGBoost'],
  'NeuralNetTorch': ['NeuralNetTorch'],
  'LightGBMLarge': ['LightG

In [76]:
y_pred_lrp_autg = autoglue_predictor_lrp.predict(X_test)
record_regression_results(model_results, 'LRP', 'Autogluon_Tab', y_test_lrp, y_pred_lrp_autg)


Test R2 score 0.4835
Test MAE score 0.0973
Test MSE score 0.0178
Test P95 score 0.2832
Test ECE score 0.0132


{'r2': 0.48345580726940873,
 'mae': 0.09733253194370031,
 'mse': 0.017826301037739564,
 'p95': 0.28319482207298274,
 'ece': 0.013173243407127761}

##### Images

In [77]:
X_train_img = pd.merge(X_train, img_path, left_index=True, right_index=True)
X_test_img = pd.merge(X_test, img_path, how="left", right_index=True, left_index=True)
train_lrp_img = pd.merge(X_train_img, y_train_lrp, left_index=True, right_index=True)

In [78]:
#autoglue_model_lrp_img = MultiModalPredictor(label="lrp", problem_type="regression", eval_metric="r2", path="../models/assessors/autoglue_tab/lrp/img/")
hyperparams = {
    "env.strategy": "ddp_notebook",
    "env.num_gpus": 1,
    "env.num_workers": 0,
    #"optim.max_epochs": 15,
    #"optim.patience": 3,
    #"optim.learning_rate": 1e-4,
    #"optim.weight_decay": 1e-4,
    #"optim.warmup_steps": 500,
    #"env.per_gpu_batch_size": 8,
    #"env.batch_size": 128,
    #"model.timm_image.checkpoint_name": "convnext_base",
}
#autoglue_predictor_lrp_img = autoglue_model_lrp_img.fit(train_lrp_img, hyperparameters=hyperparams)

In [79]:
#autoglue_predictor_lrp_img.fit_summary(verbosity=2)

In [80]:
#y_pred = autoglue_predictor_lrp_img.predict(X_test_img)
#record_regression_results(model_results, 'LRP', 'Autogluon_Mult', y_test_lrp, y_pred)


### Save predictions

In [84]:
test_inst = X_test.copy()
print(f"Number of test instances: {test_inst.shape[0]}")
metrics = ["iou", "lrp"]
models = ["baseline", "lr", "dt", "rf", "mlp", "xgb", "autg"]
for metric in metrics:
    for model in models:
        if metric == "iou":
            test_inst["GT"] = y_test
        else:
            test_inst["GT"] = y_test_lrp
        var_name = f"y_pred_{metric}_{model}"
        test_inst[model] = globals()[var_name]
    test_inst.to_csv(f"../results/assessors/{metric}_test_ass_" + data_path_file)


Number of test instances: 4035


# Final Model Comparison


In [82]:

results_df = pd.DataFrame(model_results, index=None)
results_df["test_r2"] = results_df["test_r2"].round(4)
results_df["test_mae"] = results_df["test_mae"].round(4)
#results_df["test_mse"] = results_df["test_mse"].round(4)
results_df.to_csv("../results/assessors/test_ass_results_table.csv")
display(results_df)


,target,model,test_r2,test_mae,test_ece
0,IoU,Constant Mean Predictor,-0.0005,0.1601,0.028199
1,IoU,Univariate Linear Regression (Confidence),0.2856,0.1373,0.044890
2,IoU,Linear Regression,0.4041,0.1238,0.029121
3,IoU,Decision Tree,0.4124,0.1194,0.026070
4,IoU,Random Forest,0.4335,0.1173,0.023472
5,IoU,MLP,0.3960,0.1219,0.027111
6,IoU,XGBoost,0.4168,0.1194,0.022306
7,IoU,Autogluon_Tab,0.4466,0.1167,0.020365
8,LRP,Constant Mean Predictor,-0.0000,0.1391,0.010674
9,LRP,Univariate Linear Regression (Confidence),0.2846,0.1190,0.043032


In [83]:
print(results_df.to_latex(index=False))

\begin{tabular}{llrrr}
\toprule
target & model & test_r2 & test_mae & test_ece \\
\midrule
IoU & Constant Mean Predictor & -0.000500 & 0.160100 & 0.028199 \\
IoU & Univariate Linear Regression (Confidence) & 0.285600 & 0.137300 & 0.044890 \\
IoU & Linear Regression & 0.404100 & 0.123800 & 0.029121 \\
IoU & Decision Tree & 0.412400 & 0.119400 & 0.026070 \\
IoU & Random Forest & 0.433500 & 0.117300 & 0.023472 \\
IoU & MLP & 0.396000 & 0.121900 & 0.027111 \\
IoU & XGBoost & 0.416800 & 0.119400 & 0.022306 \\
IoU & Autogluon_Tab & 0.446600 & 0.116700 & 0.020365 \\
LRP & Constant Mean Predictor & -0.000000 & 0.139100 & 0.010674 \\
LRP & Univariate Linear Regression (Confidence) & 0.284600 & 0.119000 & 0.043032 \\
LRP & Linear Regression & 0.407500 & 0.107400 & 0.022010 \\
LRP & Decision Tree & 0.442800 & 0.100800 & 0.020980 \\
LRP & Random Forest & 0.470300 & 0.098400 & 0.016344 \\
LRP & MLP & 0.448400 & 0.104800 & 0.035913 \\
LRP & XGBoost & 0.453300 & 0.099800 & 0.015282 \\
LRP & Autogluon